# 05. Model Training, Calibration, Evaluation & Explainability
## PKL Analytics & Intelligence Platform
**Role:** Senior Data Scientist / Machine Learning Engineer  
**Dataset:** Pro Kabaddi League (Seasons 1–7)

---

### Objectives
1. **Model Suite:** Train and compare Baseline (Dummy), Logistic Regression (L2 regularized), Random Forest, and HistGradientBoosting classifiers.
2. **Probability Calibration:** Apply Platt Scaling (Sigmoid Calibrated Classifier CV) on the validation set to produce reliable win probabilities.
3. **Comprehensive Evaluation:** Score all models on the unseen holdout test set (Season 7) across Accuracy, Balanced Accuracy, F1-Score, ROC-AUC, and Brier Score Loss.
4. **Interpretability:** Extract global feature importances via permutation importance and explain individual match upset drivers.
5. **Pre-Match Inference Demo:** Demonstrate real-time match forecasting with `MatchPredictor`.


In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Ensure project root in sys.path
sys.path.insert(0, os.path.abspath(".."))

from src.models.train_model import train_all_models
from src.models.evaluate_model import evaluate_models
from src.models.predict import MatchPredictor


### 1. Execute Training Pipeline and Model Selection


In [ ]:
# Train all models and calibrate best candidate
train_results = train_all_models(models_dir="../models", processed_dir="../data/processed")
print("Selected Best Model Candidate:", train_results["best_model_name"])
print("Validation Performance:")
for name, sc in train_results["val_scores"].items():
    print(f"  {name:25s} | Val Acc: {sc['val_accuracy']:.3f} | Val AUC: {sc['val_roc_auc']:.3f} | Brier: {sc['val_brier']:.3f}")


### 2. Comprehensive Evaluation on Holdout Test Set (Season 7)


In [ ]:
eval_results = evaluate_models(models_dir="../models", processed_dir="../data/processed", figures_dir="../reports/figures")

test_df_metrics = pd.DataFrame(eval_results["test_metrics"]).T
cols_display = ["accuracy", "balanced_accuracy", "f1", "roc_auc", "brier_score", "log_loss"]
test_df_metrics[cols_display].sort_values(by="roc_auc", ascending=False)


### 3. Diagnostic Figures: ROC Curves & Calibration Diagrams


In [ ]:
from IPython.display import Image, display

print("--- ROC Curves (Holdout Test Set) ---")
display(Image(filename="../reports/figures/model_roc_curves.png"))

print("--- Probability Calibration Diagram ---")
display(Image(filename="../reports/figures/model_calibration_curves.png"))

print("--- Confusion Matrices ---")
display(Image(filename="../reports/figures/model_confusion_matrices.png"))


### 4. Permutation Feature Importance


In [ ]:
top_feats = pd.DataFrame(eval_results["top_features"])
print("Top 10 Features by Permutation Importance:")
display(top_feats)

display(Image(filename="../reports/figures/model_feature_importance.png"))


### 5. Error & Upset Analysis: Why Did Underdogs Win?
Analyzing matches where the model assigned highest win probability to the losing team:


In [ ]:
upsets_df = pd.DataFrame(eval_results["top_upsets"])
print("Top Season 7 Surprises / Upsets:")
display(upsets_df)


### 6. Interactive Pre-Match Prediction Demonstration


In [ ]:
predictor = MatchPredictor(models_dir="../models", processed_dir="../data/processed")

# Predict marquee clash: Bengaluru Bulls (1) vs Dabang Delhi K.C. (2)
prediction = predictor.predict_match(team_a_id=1, team_b_id=2, as_of_date_str="2019-10-15")

print("=" * 60)
print(f"PREDICTED OUTCOME: {prediction['team_a_name']} ({prediction['team_a_prob']*100:.1f}%) vs {prediction['team_b_name']} ({prediction['team_b_prob']*100:.1f}%)")
print(f"FAVORITE: {prediction['predicted_winner']} [{prediction['confidence_tier']}]")
print("=" * 60)

print("
--- Tale of the Tape ---")
display(pd.DataFrame(prediction['tale_of_the_tape']))

print("
--- Head to Head Record ---")
print(prediction['h2h_summary'])
